In [1]:
import torch
from datasets import load_dataset
from transformer_lens import HookedTransformer
from sae_lens import SAE, HookedSAETransformer

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
# model = HookedTransformer.from_pretrained("gpt2-small", device=device)

# # the cfg dict is returned alongside the SAE since it may contain useful information for analysing the SAE (eg: instantiating an activation store)
# # Note that this is not the same as the SAEs config dict, rather it is whatever was in the HF repo, from which we can extract the SAE config dict
# # We also return the feature sparsities which are stored in HF for convenience.
# sae, cfg_dict, sparsity = SAE.from_pretrained(
#     release="gpt2-small-res-jb",  # see other options in sae_lens/pretrained_saes.yaml
#     sae_id="blocks.8.hook_resid_pre",  # won't always be a hook point
#     device=device,
# )

Loaded pretrained model gpt2-small into HookedTransformer


/home/andris/src/mech_interp/.venv/lib/python3.12/site-packages/sae_lens/sae.py:145: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [2]:
model = HookedTransformer.from_pretrained_no_processing("google/gemma-2b", dtype="bfloat16", device=device)

from sae_lens import SAE

release = "gemma-2b-res-jb"
sae_id = "blocks.12.hook_resid_post"
sae = SAE.from_pretrained(release, sae_id)[0]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

In [2]:
# LLma3.2-1B
from sae_lens import SAE

model = HookedTransformer.from_pretrained_no_processing("meta-llama/Llama-3.2-1B", dtype="bfloat16", device=device) 
sae, cfg_dict, sparsity = SAE.from_pretrained("yoonLM/sae_llama3.2org_1B_512_16_l1_100", "blocks.10.hook_resid_pre", device=device)

Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer


/home/andris/src/mech_interp/.venv/lib/python3.12/site-packages/sae_lens/sae.py:145: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [3]:
from transformer_lens.utils import tokenize_and_concatenate
from datasets import load_dataset
dataset = load_dataset(
    path="NeelNanda/pile-10k",
    split="train",
    streaming=False,
)

token_dataset = tokenize_and_concatenate(
    dataset=dataset,  # type: ignore
    tokenizer=model.tokenizer,  # type: ignore
    streaming=True,
    max_length=sae.cfg.context_size,
    add_bos_token=sae.cfg.prepend_bos,
)

In [4]:
torch.cuda.empty_cache()

from sae_dashboard.sae_vis_data import SaeVisConfig
from sae_dashboard.sae_vis_runner import SaeVisRunner

test_feature_idx_gpt = list(range(10)) # + [14057]
hook_name = sae.cfg.hook_name
# Todo what does this config
feature_vis_config_gpt = SaeVisConfig(
    hook_point=hook_name,
    features=test_feature_idx_gpt,
    minibatch_size_features=4,
    minibatch_size_tokens=16,
    verbose=True,
    device=device,
)

with torch.no_grad():
    visualization_data_gpt = SaeVisRunner(feature_vis_config_gpt).run(
        encoder=sae,   # type: ignore
        model=model,
        tokens=token_dataset[:10000]["tokens"],   # type: ignore
    )


Forward passes to cache data for vis:   0%|          | 0/1875 [00:00<?, ?it/s]

Extracting vis data from cached data:   0%|          | 0/10 [00:00<?, ?it/s]

┏━━━━━━┳━━━━━━┳━━━━━━━┓
┃ Task ┃ Time ┃ Pct % ┃
┡━━━━━━╇━━━━━━╇━━━━━━━┩
└──────┴──────┴───────┘

In [5]:
from sae_dashboard.data_writing_fns import save_feature_centric_vis

filename = f"demo_feature_dashboards-LLama-3.2-1B-10.html"
save_feature_centric_vis(sae_vis_data=visualization_data_gpt, filename=filename)

Saving feature-centric vis:   0%|          | 0/10 [00:00<?, ?it/s]